Feature-based baselines (response-only TF-IDF + simple engineered features)


### Goal

Build a stronger feature-based baseline for hallucination detection using:
- TF-IDF over the **response** only (lexical signal),
- simple engineered numeric features (style/structure signals),
- a combined linear classifier pipeline.

This notebook assumes the earlier prompt+response baseline was evaluated in Notebook 02 and focuses on a cleaner response-only setup.


### Refactor notes

This notebook now uses shared utilities under `src/` for data loading, feature extraction, model construction, and evaluation.
That keeps the notebook focused on experiment flow (fit/eval/ablation/sanity checks) and results, rather than duplicated infrastructure code.


In [ ]:
import sys
from pathlib import Path

# Ensure repo root is in PYTHONPATH when running from notebooks/
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.data import load_splits
from src.features import add_numeric_feature_columns, get_numeric_feature_cols
from src.models import build_tfidf_only_logreg, build_tfidf_numeric_logreg
from src.utils import evaluate_split, metrics_table


### Load splits

We load the pre-built train/val/test splits from `data_processed/` using the shared loader.


In [ ]:
train_df, val_df, test_df = load_splits(root=ROOT)
print(train_df.shape, val_df.shape, test_df.shape)
train_df.head(2)


### Numeric feature extraction

We extract lightweight, interpretable numeric features from each response.
These features aim to capture stylistic patterns (e.g., length, punctuation usage, numeric content, uncertainty expressions) that may correlate with hallucinated outputs.


### Apply Features + Sanity Checks

We generate numeric feature columns for each split and run quick checks to confirm:
- no unexpected missing values,
- consistent feature columns across splits,
- shapes look reasonable before modeling.


In [ ]:
train_feat = add_numeric_feature_columns(train_df, response_col="response")
val_feat   = add_numeric_feature_columns(val_df, response_col="response")
test_feat  = add_numeric_feature_columns(test_df, response_col="response")

numeric_feature_cols = get_numeric_feature_cols(train_feat)
print("Numeric feature columns:", numeric_feature_cols)

# Quick sanity checks
for name, df in [("train", train_feat), ("val", val_feat), ("test", test_feat)]:
    n_nan = df[numeric_feature_cols].isna().sum().sum()
    print(f"{name}: numeric NaN count = {n_nan}, shape={df.shape}")


### Sanity Check: Feature Distributions

Before training, we visualize a small subset of feature distributions to ensure:
- features are not constant (“dead features”),
- ranges are reasonable (no extreme scaling issues),
- sparse features (e.g., numbers / uncertainty terms) still have variation.


In [ ]:
def plot_feature_histograms(df: pd.DataFrame, cols, bins=30):
    """Quick sanity-check histograms for numeric features."""
    for col in cols:
        plt.figure()
        df[col].hist(bins=bins)
        plt.title(col)
        plt.xlabel(col)
        plt.ylabel("count")
        plt.show()

plot_cols = ["resp_n_words", "resp_n_numbers", "resp_n_uncertainty", "resp_punct_per_word"]
plot_feature_histograms(train_feat, plot_cols)


### Modeling pipeline

We construct a unified modeling pipeline using `ColumnTransformer`:
- TF-IDF is applied to the **response text only** to capture lexical information.
- Numeric features are standardized and concatenated with the TF-IDF representation.

The combined feature space is fed into a Logistic Regression classifier, serving as a strong and interpretable baseline.


### Training and Evaluation

We train on the fixed training split and evaluate on validation/test using Accuracy, Precision, Recall, and F1-score. F1 is emphasized because it balances false positives and false negatives, which is important for hallucination detection.


In [ ]:
pipe_only = build_tfidf_only_logreg()
pipe_full = build_tfidf_numeric_logreg(numeric_feature_cols)

pipe_only.fit(train_feat, train_feat["label"])
pipe_full.fit(train_feat, train_feat["label"])

val_only = evaluate_split("val", pipe_only, val_feat, val_feat["label"])
val_full = evaluate_split("val", pipe_full, val_feat, val_feat["label"])
test_full = evaluate_split("test", pipe_full, test_feat, test_feat["label"])


### Ablation Study

To quantify the contribution of engineered numeric features, we compare:
1) **TF-IDF(response) only**
2) **TF-IDF(response) + numeric features**

A consistent improvement indicates that the numeric features add complementary information beyond lexical content.


In [ ]:
metrics_table([
    {"model": "TF-IDF(response) only", "split": "val", **val_only.as_dict()},
    {"model": "TF-IDF(response)+numeric", "split": "val", **val_full.as_dict()},
    {"model": "TF-IDF(response)+numeric", "split": "test", **test_full.as_dict()},
])


### Sanity Checks (No Leakage)

We run additional checks to validate the experimental setup:
- **Train vs. validation/test gap** to assess generalization,
- **Shuffled-label test** (expected ~random performance) to rule out leakage,
- **Overlap checks** across splits to identify potential duplicates.


In [ ]:
from sklearn.utils import shuffle

y_train_shuffled = shuffle(train_feat["label"].astype(int), random_state=42)

pipe_shuf = build_tfidf_numeric_logreg(numeric_feature_cols)
pipe_shuf.fit(train_feat, y_train_shuffled)

_ = evaluate_split("val (shuffled labels)", pipe_shuf, val_feat, val_feat["label"])


In [ ]:
def overlap_rate(a: pd.Series, b: pd.Series) -> float:
    a_set = set(a.fillna("").astype(str))
    b_set = set(b.fillna("").astype(str))
    inter = a_set.intersection(b_set)
    return len(inter) / max(len(b_set), 1)

print("Response overlap train->val:", overlap_rate(train_df["response"], val_df["response"]))
print("Response overlap train->test:", overlap_rate(train_df["response"], test_df["response"]))

# Stronger check: exact (prompt, response) pairs
train_pairs = set(zip(train_df["prompt"].fillna("").astype(str), train_df["response"].fillna("").astype(str)))
val_pairs   = set(zip(val_df["prompt"].fillna("").astype(str),   val_df["response"].fillna("").astype(str)))
test_pairs  = set(zip(test_df["prompt"].fillna("").astype(str),  test_df["response"].fillna("").astype(str)))

print("Exact pair overlap train->val:", len(train_pairs & val_pairs) / max(len(val_pairs), 1))
print("Exact pair overlap train->test:", len(train_pairs & test_pairs) / max(len(test_pairs), 1))


### Summary and discussion

This notebook evaluated a feature-based baseline for hallucination detection using response-only representations.

**Baseline reference:**  
An earlier TF-IDF baseline using both prompt and response text was evaluated in **Notebook 02**, yielding near-random performance (≈0.49 accuracy / F1 on validation).
That experiment is not reproduced here.

By restricting the representation to the generated response and combining lexical (TF-IDF) and stylistic (numeric) features, we observe a substantial and stable improvement in performance.

An ablation study shows that numeric features provide a small but consistent gain over TF-IDF alone, indicating that stylistic cues add complementary information beyond lexical content.

Additional sanity checks (train–validation gap, shuffled labels, overlap analysis) confirm that the results are not driven by data leakage.
